# P7-1 · Bài 9 · ViT và CLIP

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import torch, torch.nn as nn

class ViT(nn.Module):
    def __init__(self, p=2, d=64, depth=4, heads=4, img=8, n_cls=10):
        super().__init__()
        self.patch = nn.Conv2d(1, d, p, stride=p)                       # patch embedding
        n = (img // p) ** 2
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        self.pos = nn.Parameter(torch.randn(1, n + 1, d) * .02)         # embedding vị trí
        layer = nn.TransformerEncoderLayer(d, heads, 2 * d, dropout=0.1, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, depth)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, n_cls))
    def forward(self, x):
        t = self.patch(x).flatten(2).transpose(1, 2)                     # (B, n, d)
        t = torch.cat([self.cls.expand(len(t), -1, -1), t], 1) + self.pos
        return self.head(self.enc(t)[:, 0])                              # lấy đầu ra của CLS

vit = ViT()
x = torch.randn(5, 1, 8, 8)                                   # 5 ảnh xám 8×8
print(vit.patch(x).flatten(2).transpose(1, 2).shape)          # 16 mảnh 2×2, mỗi mảnh 64 chiều
print(vit(x).shape, sum(p.numel() for p in vit.parameters()), "tham số")

torch.Size([5, 16, 64])
torch.Size([5, 10]) 136138 tham số


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
# emb_train, emb_test: embedding ảnh từ encoder CLIP đã đóng băng, chuẩn hóa độ dài 1.
# Chạy thử ở đây: thay CLIP bằng một "encoder đóng băng" đơn giản (PCA) trên chữ số viết tay
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
X, y = load_digits(return_X_y=True)
X_tr, X_te, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
enc = PCA(32).fit(X_tr)
norm = lambda E: E / np.linalg.norm(E, axis=1, keepdims=True)
emb_train, emb_test = norm(enc.transform(X_tr)), norm(enc.transform(X_te))

clf = LogisticRegression(C=1.0, max_iter=2000).fit(emb_train, y_train)
print("accuracy:", clf.score(emb_test, y_test))

accuracy: 0.9407407407407408


## Bài tập 5: Zero-shot với transformers {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
# Cần mạng: tải CLIP từ Hugging Face và ảnh của bạn, chạy trên Colab
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16").eval()
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")
TEMPLATES = ["a photo of a {}.", "a product photo of a {}.", "a close-up photo of a {}."]

@torch.no_grad()
def class_embeddings(classes):
    out = []
    for c in classes:
        t = proc(text=[tpl.format(c) for tpl in TEMPLATES], return_tensors="pt", padding=True)
        e = model.get_text_features(**t)
        e = e / e.norm(dim=-1, keepdim=True)
        out.append(e.mean(0) / e.mean(0).norm())
    return torch.stack(out)

@torch.no_grad()
def zero_shot(image_path, classes):
    img = model.get_image_features(**proc(images=Image.open(image_path).convert("RGB"), return_tensors="pt"))
    img = img / img.norm(dim=-1, keepdim=True)
    probs = (100 * img @ class_embeddings(classes).T).softmax(-1)[0]
    return dict(zip(classes, probs.tolist()))